In [ ]:
# CONSTANTS

IBKR_PORT      = 7496

INPUT_WB_NAME  = "2026 Stat Arb Mkt Data.xlsx"
INPUT_WS_NAME  = "MAIN"
INPUT_TBL_NAME = "C3:G63"

In [2]:
import pandas as pd

In [3]:
# --- system setup ---
import os
import sys
sys.path.append(os.path.abspath(".."))

In [4]:
# --- utils ---
# from IPython.display import display, clear_output
# from input_output.Standard_Input_and_Output import standard_input, standard_output
from input_output.Class_InputOutput import InputOutput
io = InputOutput()

In [5]:
# --- IBKR ---
from ib_insync import Contract, Order
from ibkr.Class_IBKR_IB import IBKR_IB
# from ibkr.Class_IBKR_TWS import IBKR_TWS
ibkr = IBKR_IB(port=IBKR_PORT)

In [ ]:

wb, ws = io.set_xw_book_and_sheet(INPUT_WB_NAME, INPUT_WS_NAME)
df = io.get_xw_df(ws, INPUT_TBL_NAME, table=False, headerRows=1, style=float)
df['order'] = df['order'].astype('boolean').ffill()
df = df[df['order'] == True]
df = df[['symbol', 'diff']]

,symbol,diff
2,PCY,2715.069262
3,EMLC,-2268.757470
4,CWB,-727.734591
5,ICVT,651.212514
14,VTEB,3178.603082
15,MUB,-1494.121818
20,BWX,-4590.314437
21,IGOV,2426.798720


In [7]:
await ibkr.connect()

for row in df.itertuples(index=False):   

    if pd.isna(row.diff) or row.diff == 0:
        continue

    contract = Contract(
        symbol=row.symbol,
        secType="STK",
        exchange="SMART",
        currency="USD",
    )

    qualified = await ibkr.ib.qualifyContractsAsync(contract)

    if not qualified:
        print(f"Could not qualify {row.symbol}")
        continue

    contract = qualified[0]

    buy_sell = "BUY" if row.diff > 0 else "SELL"
    size = abs(round(row.diff))

    if size == 0:
        continue

    order = Order(
        action=buy_sell,
        orderType="MOC",
        totalQuantity=size,
        tif="DAY",
    )

    trade = ibkr.ib.placeOrder(contract, order)

    print(trade)


Trade(contract=Contract(secType='STK', conId=319355940, symbol='PCY', exchange='SMART', primaryExchange='ARCA', currency='USD', localSymbol='PCY', tradingClass='PCY'), order=Order(orderId=4, clientId=154746, action='BUY', totalQuantity=2715, orderType='MOC', tif='DAY'), orderStatus=OrderStatus(orderId=4, status='PendingSubmit', filled=0.0, remaining=0.0, avgFillPrice=0.0, permId=0, parentId=0, lastFillPrice=0.0, clientId=0, whyHeld='', mktCapPrice=0.0), fills=[], log=[TradeLogEntry(time=datetime.datetime(2026, 7, 30, 19, 47, 46, 878150, tzinfo=datetime.timezone.utc), status='PendingSubmit', message='', errorCode=0)], advancedError='')
Trade(contract=Contract(secType='STK', conId=337332930, symbol='EMLC', exchange='SMART', primaryExchange='ARCA', currency='USD', localSymbol='EMLC', tradingClass='EMLC'), order=Order(orderId=6, clientId=154746, action='SELL', totalQuantity=2269, orderType='MOC', tif='DAY'), orderStatus=OrderStatus(orderId=6, status='PendingSubmit', filled=0.0, remaining=0